# 11 — ORPO using TRL

## Learning objectives

- Inspect pinned trainer signatures and tokenized preference batches.
- Run a no-download two-step CPU trainer smoke test.
- Save/evaluate/reload and optionally fine-tune a pretrained model.

This notebook runs a small CPU demonstration by default. Synthetic experiments are educational, not evidence of LLM alignment. Pretrained-model downloads and GPU training require explicit opt-in. All runtime outputs stay inside `RLHF_NOTEBOOKS/`.

**Source connection:** ORPO source notebook walkthroughs and corrected notes Chapters07/09. Exact TRL0.22.2 wheel source retained in api_references; runtime signatures are checked.

[Learning order and setup](README.md).

In [1]:
from pathlib import Path
import os, sys, json, math, random, copy, re, subprocess, hashlib
# Find this collection when launched from the repository, module, or notebook directory.
candidates = [Path.cwd(), Path.cwd()/'RLHF_NOTEBOOKS', Path.cwd()/'13_rlhf_ppo_dpo_orpo/RLHF_NOTEBOOKS']
ROOT = next((p.resolve() for p in candidates if (p/'source_inspection.json').exists()), None)
if ROOT is None:
    raise RuntimeError('Launch Jupyter from RLHF_NOTEBOOKS, its parent, or the repository root.')
RUNTIME = ROOT/'runtime'; RUNTIME.mkdir(exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(RUNTIME/'matplotlib')
os.environ['HF_HOME'] = str(RUNTIME/'huggingface')
os.environ['HF_DATASETS_CACHE'] = str(RUNTIME/'huggingface/datasets')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
random.seed(42); np.random.seed(42); torch.manual_seed(42)
torch.set_num_threads(1)
print('Python', sys.version.split()[0], 'PyTorch', torch.__version__, 'CUDA', torch.cuda.is_available())

Python 3.13.9 PyTorch 2.8.0+cpu CUDA False


## Version and behavior boundary

This recipe targets TRL0.22.2, Transformers4.56.2, Datasets4.3.0, PEFT0.17.1 and Accelerate1.10.1. It uses `processing_class`, not a version-mixed `tokenizer` argument. The CPU model is a random small GPT-2 architecture; it cannot demonstrate useful alignment. The pretrained path is off by default. Read notebook10 for explicit math before this abstraction. ORPO combines supervised and odds preference objectives without a reference.

In [2]:
def read_preferences():
    path = ROOT.parent/'reward_preferences_dataset.jsonl'
    if path.exists():
        raw = path.read_bytes(); origin = str(path)
    else:
        # Originals are currently missing from the working tree. Read their Git blob;
        # never restore, overwrite or silently substitute invented preference records.
        gitroot = Path(subprocess.check_output(['git','rev-parse','--show-toplevel'], cwd=ROOT, text=True).strip())
        rel = (ROOT.parent/'reward_preferences_dataset.jsonl').relative_to(gitroot).as_posix()
        raw = subprocess.check_output(['git','show','HEAD:'+rel], cwd=gitroot)
        origin = 'HEAD:'+rel
    expected = json.loads((ROOT/'source_inspection.json').read_text())
    digest = next(r['sha256'] for r in expected if r['source']=='reward_preferences_dataset.jsonl')
    assert hashlib.sha256(raw).hexdigest() == digest, 'Dataset changed: re-audit before proceeding'
    rows = [json.loads(line) for line in raw.decode('utf-8').splitlines()]
    for row in rows:
        assert set(row)=={'prompt','chosen','rejected'}
        assert all(isinstance(row[k],str) and row[k].strip() for k in row)
        assert row['chosen'].strip()!=row['rejected'].strip()
    assert len({r['prompt'].strip().lower() for r in rows})==len(rows)
    print('Dataset origin:',origin,'records:',len(rows),'fields:',list(rows[0]))
    return rows
rows = read_preferences()
indices = np.random.default_rng(42).permutation(len(rows))
train_rows = [rows[i] for i in indices[:14]]
valid_rows = [rows[i] for i in indices[14:17]]
test_rows = [rows[i] for i in indices[17:]]
assert not ({r['prompt'] for r in train_rows} & {r['prompt'] for r in valid_rows+test_rows})
print('Prompt-separated split:',len(train_rows),len(valid_rows),len(test_rows))

Dataset origin: HEAD:13_rlhf_ppo_dpo_orpo/reward_preferences_dataset.jsonl records: 20 fields: ['prompt', 'chosen', 'rejected']
Prompt-separated split: 14 3 3


In [3]:
# Local tiny GPT-2 architecture and WordLevel tokenizer: NO pretrained downloads.
from importlib import metadata
import inspect
from tokenizers import Tokenizer,models,pre_tokenizers
from transformers import PreTrainedTokenizerFast,GPT2Config,GPT2LMHeadModel,GPT2ForSequenceClassification
from datasets import Dataset
from trl import DPOTrainer,DPOConfig,GRPOTrainer,GRPOConfig,ORPOTrainer,ORPOConfig,RewardTrainer,RewardConfig,PPOTrainer,PPOConfig
VERSIONS={name:metadata.version(name) for name in ('torch','transformers','trl','datasets','peft','accelerate')}
print(VERSIONS)
assert metadata.version('trl')=='0.22.2', 'Use the pinned environment; APIs change across TRL releases.'
for cls in [DPOTrainer,GRPOTrainer,ORPOTrainer,PPOTrainer]: print(cls.__name__,inspect.signature(cls.__init__))
texts=['Question: '+r['prompt']+' Answer: '+r[s] for r in rows for s in ('chosen','rejected')]
words=sorted(set(re.findall(r'\w+|[^\w\s]', ' '.join(texts)+' What is 1 + 1 ? 2 3 4 - .')))
word_vocab={'<pad>':0,'<eos>':1,'<unk>':2,'<bos>':3,**{w:i+4 for i,w in enumerate(words)}}
backend=Tokenizer(models.WordLevel(word_vocab,unk_token='<unk>'));backend.pre_tokenizer=pre_tokenizers.Whitespace()
tokenizer=PreTrainedTokenizerFast(tokenizer_object=backend,pad_token='<pad>',eos_token='<eos>',unk_token='<unk>',bos_token='<bos>')
tokenizer.chat_template="{% for message in messages %}{{ message['role'] + ': ' + message['content'] + '\\n' }}{% endfor %}{% if add_generation_prompt %}{{ 'assistant: ' }}{% endif %}"
def lm_config():
    return GPT2Config(vocab_size=len(tokenizer),n_positions=256,n_ctx=256,n_embd=32,n_layer=1,n_head=2,pad_token_id=0,eos_token_id=1,bos_token_id=3,resid_pdrop=0.,embd_pdrop=0.,attn_pdrop=0.)
def hf_pairs(examples):
    return Dataset.from_list([{'prompt':'Question: '+r['prompt']+' Answer:','chosen':' '+r['chosen']+tokenizer.eos_token,'rejected':' '+r['rejected']+tokenizer.eos_token} for r in examples])
RUN_TRL_SMOKE = True   # two steps on randomly initialized tiny LOCAL model, CPU-friendly
RUN_PRETRAINED_GPU = False  # opt-in; downloads a real checkpoint only when True
print('Tokenizer distinct PAD/EOS:',tokenizer.pad_token_id,tokenizer.eos_token_id)
assert tokenizer.pad_token_id != tokenizer.eos_token_id

/home/akhand/Desktop/LLM FINETUNING/SLM-Lab-Small-Language-Model-Experiments/13_rlhf_ppo_dpo_orpo/RLHF_NOTEBOOKS/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


{'torch': '2.8.0+cpu', 'transformers': '4.56.2', 'trl': '0.22.2', 'datasets': '4.3.0', 'peft': '0.17.1', 'accelerate': '1.10.1'}
DPOTrainer (self, model: Union[str, torch.nn.modules.module.Module, transformers.modeling_utils.PreTrainedModel], ref_model: Union[transformers.modeling_utils.PreTrainedModel, torch.nn.modules.module.Module, str, NoneType] = None, args: Optional[trl.trainer.dpo_config.DPOConfig] = None, data_collator: Optional[transformers.data.data_collator.DataCollator] = None, train_dataset: Union[datasets.arrow_dataset.Dataset, datasets.iterable_dataset.IterableDataset, NoneType] = None, eval_dataset: Union[datasets.arrow_dataset.Dataset, datasets.iterable_dataset.IterableDataset, dict[str, Union[datasets.arrow_dataset.Dataset, datasets.iterable_dataset.IterableDataset]], NoneType] = None, processing_class: Union[transformers.tokenization_utils_base.PreTrainedTokenizerBase, transformers.image_processing_utils.BaseImageProcessor, transformers.feature_extraction_utils.Featu

In [4]:
if RUN_TRL_SMOKE:
    model=GPT2LMHeadModel(lm_config())
    args=ORPOConfig(output_dir=str(RUNTIME/'orpo_cpu'),max_steps=2,learning_rate=1e-4,per_device_train_batch_size=2,per_device_eval_batch_size=2,
                gradient_accumulation_steps=1,max_length=128,max_prompt_length=64,beta=.1,
                eval_strategy='steps',eval_steps=1,save_strategy='steps',save_steps=1,save_total_limit=2,
                logging_steps=1,report_to='none',use_cpu=True,bf16=False,fp16=False,remove_unused_columns=False,seed=42)
    trainer=ORPOTrainer(model=model,args=args,processing_class=tokenizer,train_dataset=hf_pairs(train_rows),eval_dataset=hf_pairs(valid_rows))
    initial=trainer.evaluate();result=trainer.train();final=trainer.evaluate()
    trainer.save_model(str(RUNTIME/'orpo_cpu/final'));tokenizer.save_pretrained(str(RUNTIME/'orpo_cpu/final'))
    print('Actual local smoke output:',result,'Before:',initial,'After:',final)
    # Reload local model; no pretrained download; compare logits on exactly same input.
    trained=trainer.accelerator.unwrap_model(trainer.model).eval()
    reloaded=GPT2LMHeadModel.from_pretrained(str(RUNTIME/'orpo_cpu/final')).eval()
    x=tokenizer('Question: What is AI? Answer:',return_tensors='pt')
    with torch.no_grad():assert torch.allclose(trained(**x).logits,reloaded(**x).logits,atol=1e-5)
    print('Saved/reloaded logits agree.')

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Map: 100%|██████████| 14/14 [00:00<00:00, 5768.76 examples/s]

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Map: 100%|██████████| 14/14 [00:00<00:00, 7317.17 examples/s]

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

...

Step,Training Loss,Validation Loss,Model Preparation Time,Runtime,Samples Per Second,Steps Per Second,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/rejected,Logps/chosen,Logits/rejected,Logits/chosen,Nll Loss,Log Odds Ratio,Log Odds Chosen
1,5.844100,5.867369,0.000200,0.008700,345.381000,230.254000,-0.579251,-0.578500,0.750000,-0.000751,-5.785002,-5.792507,0.003294,0.002141,5.794736,-0.697070,-0.007529
2,5.834500,5.865788,0.000200,0.008400,357.246000,238.164000,-0.579193,-0.578466,0.750000,-0.000727,-5.784664,-5.791935,0.003128,0.001931,5.793126,-0.696954,-0.007293


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


/home/akhand/Desktop/LLM FINETUNING/SLM-Lab-Small-Language-Model-Experiments/13_rlhf_ppo_dpo_orpo/RLHF_NOTEBOOKS/.venv/lib/python3.13/site-packages/transformers/trainer.py:4360: UserWarning: mtime may not be reliable on this filesystem, falling back to numerical ordering
  warnings.warn("mtime may not be reliable on this filesystem, falling back to numerical ordering")


Actual local smoke output: TrainOutput(global_step=2, training_loss=5.839309453964233, metrics={'train_runtime': 0.0617, 'train_samples_per_second': 64.786, 'train_steps_per_second': 32.393, 'total_flos': 0.0, 'train_loss': 5.839309453964233, 'epoch': 0.2857142857142857}) Before: {'eval_loss': 5.8707146644592285, 'eval_model_preparation_time': 0.0002, 'eval_runtime': 0.0123, 'eval_samples_per_second': 244.224, 'eval_steps_per_second': 162.816, 'eval_rewards/chosen': -0.5794106721878052, 'eval_rewards/rejected': -0.5785900354385376, 'eval_rewards/accuracies': 0.75, 'eval_rewards/margins': -0.0008206367492675781, 'eval_logps/rejected': -5.785900115966797, 'eval_logps/chosen': -5.794106483459473, 'eval_logits/rejected': 0.0036767940036952496, 'eval_logits/chosen': 0.002638735808432102, 'eval_nll_loss': 5.798244476318359, 'eval_log_odds_ratio': -0.6974189281463623, 'eval_log_odds_chosen': -0.00823196955025196} After: {'eval_loss': 5.865787982940674, 'eval_model_preparation_time': 0.0002, '

In [5]:
if RUN_TRL_SMOKE:
    batch=next(iter(trainer.get_train_dataloader()))
    print('Actual collated keys/shapes:',{k:tuple(v.shape) if hasattr(v,'shape') else type(v).__name__ for k,v in batch.items()})
    for k,v in batch.items():
        if isinstance(v,torch.Tensor) and k.endswith('input_ids'):
            assert v.min()>=0 and v.max()<len(tokenizer)
        if isinstance(v,torch.Tensor) and k.endswith('labels'):
            active=v.ne(-100);assert active.any()
    checkpoints=sorted((RUNTIME/('dpo_cpu' if isinstance(trainer,DPOTrainer) else 'orpo_cpu')).glob('checkpoint-*'))
    print('Full trainer checkpoints:',[p.name for p in checkpoints])
    # To resume training: trainer.train(resume_from_checkpoint=str(checkpoints[-1]))
    # Model-only final export is not optimizer/RNG state; resume from checkpoint-*.

Actual collated keys/shapes: {'prompt': 'list', 'chosen': 'list', 'rejected': 'list', 'chosen_input_ids': (2, 32), 'chosen_attention_mask': (2, 32), 'chosen_labels': (2, 32), 'rejected_input_ids': (2, 27), 'rejected_attention_mask': (2, 27), 'rejected_labels': (2, 27), 'prompt_input_ids': (2, 13), 'prompt_token_type_ids': 'list', 'prompt_attention_mask': (2, 13)}
Full trainer checkpoints: ['checkpoint-1', 'checkpoint-2']


## Real-model template and memory workflow

Use structured user/assistant messages, template exactly once, inspect truncation before training, mask prompts/padding, and evaluate disjoint prompts. Quantization and PEFT are distinct: this small GPU recipe uses LoRA and supported precision without bitsandbytes. No device_map="auto" is used for training. Five optimizer steps are only a smoke run; enlarge data and evaluate task quality before any alignment claim.

In [6]:
# Real pretrained-model path; this block is skipped by default.
if RUN_PRETRAINED_GPU:
    assert torch.cuda.is_available(), 'This optional recipe requires CUDA; leave the flag False for CPU demos.'
    from transformers import AutoTokenizer,AutoModelForCausalLM
    from peft import LoraConfig,AutoPeftModelForCausalLM
    MODEL_ID='HuggingFaceTB/SmolLM2-135M-Instruct'
    gpu_tok=AutoTokenizer.from_pretrained(MODEL_ID)
    if gpu_tok.pad_token_id is None:gpu_tok.pad_token=gpu_tok.eos_token
    # Chat templates own formatting. Do not apply them and then template again in trainer.
    def conversational(examples):
        return Dataset.from_list([{'prompt':[{'role':'user','content':r['prompt']}],
                                  'chosen':[{'role':'assistant','content':r['chosen']}],
                                  'rejected':[{'role':'assistant','content':r['rejected']}]} for r in examples])
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
    gpu_model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=dtype)
    gpu_model.config.use_cache=False
    config=ORPOConfig(output_dir=str(RUNTIME/'orpo_gpu'),max_steps=5,per_device_train_batch_size=1,gradient_accumulation_steps=4,
                  per_device_eval_batch_size=1,learning_rate=1e-5,beta=.1,max_length=256,max_prompt_length=128,
                  bf16=dtype==torch.bfloat16,fp16=False,eval_strategy='steps',eval_steps=5,save_strategy='steps',save_steps=5,
                  gradient_checkpointing=True,report_to='none',seed=42,remove_unused_columns=False)
    gpu_trainer=ORPOTrainer(model=gpu_model,args=config,processing_class=gpu_tok,train_dataset=conversational(train_rows),
                        eval_dataset=conversational(valid_rows),peft_config=LoraConfig(task_type='CAUSAL_LM',r=8,lora_alpha=16,target_modules=['q_proj','v_proj']))
    gpu_trainer.train();print(gpu_trainer.evaluate());gpu_trainer.save_model(str(RUNTIME/'orpo_gpu/final'));gpu_tok.save_pretrained(str(RUNTIME/'orpo_gpu/final'))
    # DPO ref_model=None with a new LoRA adapter still uses adapter-disabled reference.
    # ORPO has no reference role. Do not interpret identical keyword choices as identical objectives.
    del gpu_trainer,gpu_model
    loaded=AutoPeftModelForCausalLM.from_pretrained(str(RUNTIME/'orpo_gpu/final'),torch_dtype=dtype).to('cuda').eval()
    inputs=gpu_tok.apply_chat_template([{'role':'user','content':'What is AI?'}],tokenize=True,add_generation_prompt=True,return_tensors='pt').to('cuda')
    with torch.no_grad():generated=loaded.generate(inputs,max_new_tokens=48,do_sample=False,pad_token_id=gpu_tok.pad_token_id)
    print(gpu_tok.decode(generated[0,inputs.shape[1]:],skip_special_tokens=True))

## Exercises

1. Inspect the actual labels/masks and decoded prompt boundary.
2. Reload a checkpoint and verify optimizer step/RNG state.
3. Compare train/validation preference metrics and response length against the base model.

## Key takeaways

Recompute the worked examples, inspect masks and gradients, and evaluate independently of the training objective. A small training-loss decrease does not establish generalization or alignment. Record the model, data split, seeds and dependency versions before interpreting results.